# E5 — 4-Class DR Severity (Merged No DR + Mild)

**Purpose:** Train DR severity grading with merged classes 0+1 → No Referable DR.

**Rationale:** Experiments E0–E4b proved Mild DR (class 1) is unlearnable on EyePACS:
- E0: ResNet18 + CE → QWK 0.6269, Mild recall 0%
- E3+: ResNet18 + CE (tuned) → QWK 0.6456, Mild recall 0%
- E4: EfficientNet-B3 + CE → QWK 0.5919, Mild recall 0%
- E4b: EfficientNet-B3 + EMD → QWK 0.6115, Mild recall 0%

By merging the noisy 0↔1 boundary, we expect a significant QWK boost.

**4-Class Mapping:**
| New Class | Old Classes | Name | Count |
|-----------|-------------|------|-------|
| 0 | 0 + 1 | No Referable DR | 28,253 |
| 1 | 2 | Moderate DR | 5,292 |
| 2 | 3 | Severe DR | 873 |
| 3 | 4 | Proliferative DR | 708 |

**Strategy:**
- Phase 1 (epochs 1–10): Freeze backbone, train severity head only
- Phase 2 (epochs 11–30): Unfreeze top 2 EfficientNet blocks + head
- Class-weighted Cross-Entropy to handle imbalance

---

## Data Upload Instructions

### Step 1: Run locally on your machine
```bash
python scripts/prep_4class_zip.py
```
This creates `eyepacs_4class_splits.zip` with remapped labels.

### Step 2: Upload to Google Drive
Upload `eyepacs_4class_splits.zip` to: **My Drive/eye-realtime-inference/**

### Step 3: Run this notebook
1. **Runtime → Change runtime type → T4 GPU**
2. Run all cells

## 0. GPU Check & Setup

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Install dependencies
!pip install -q scikit-learn tqdm pandas pillow seaborn

## 1. Mount Google Drive & Extract Data to Local SSD

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
import zipfile
import time

# ============================================================
# CONFIGURE THIS: path to your project folder in Google Drive
# ============================================================
DRIVE_PROJECT = "/content/drive/MyDrive/eye-realtime-inference"
ZIP_PATH = os.path.join(DRIVE_PROJECT, "eyepacs_4class_splits.zip")
LOCAL_DATA = "/content/data"  # Fast local SSD

# Verify zip exists
assert os.path.isfile(ZIP_PATH), (
    f"Zip file not found at {ZIP_PATH}\n"
    f"Upload eyepacs_4class_splits.zip to: My Drive/eye-realtime-inference/"
)
zip_size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f"Found zip: {zip_size_mb:.0f} MB")

# Extract to local SSD with Windows backslash fix
train_dir = os.path.join(LOCAL_DATA, "train")
val_dir = os.path.join(LOCAL_DATA, "val")

if not os.path.isdir(train_dir) or len(os.listdir(train_dir)) == 0:
    if os.path.isdir(LOCAL_DATA):
        shutil.rmtree(LOCAL_DATA)

    print("Extracting to local SSD (fixing Windows path separators)...")
    start = time.time()
    os.makedirs(LOCAL_DATA, exist_ok=True)

    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        for member in zf.infolist():
            fixed_name = member.filename.replace('\\', '/')
            if fixed_name.endswith('/'):
                continue
            out_path = os.path.join(LOCAL_DATA, fixed_name)
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
            with zf.open(member) as src, open(out_path, 'wb') as dst:
                shutil.copyfileobj(src, dst)

    elapsed = time.time() - start
    print(f"Extraction complete in {elapsed:.0f}s")
else:
    print("Data already extracted, skipping.")

# Auto-detect paths
def find_dir_with_images(base, candidates):
    for c in candidates:
        p = os.path.join(base, c)
        if os.path.isdir(p):
            has_images = any(f.endswith(('.jpeg', '.jpg', '.png')) for f in os.listdir(p))
            has_subdirs = any(os.path.isdir(os.path.join(p, d)) for d in os.listdir(p))
            if has_images or has_subdirs:
                return p
    return None

train_dir = find_dir_with_images(LOCAL_DATA, ["train", "eyepacs/train", "Data/splits/fundus/eyepacs/train"])
val_dir = find_dir_with_images(LOCAL_DATA, ["val", "eyepacs/val", "Data/splits/fundus/eyepacs/val"])

# Find 4-class labels CSV
labels_csv = None
for c in ["eyepacs_4class_trainLabels.csv", "labels/eyepacs_4class_trainLabels.csv"]:
    p = os.path.join(LOCAL_DATA, c)
    if os.path.isfile(p):
        labels_csv = p
        break

if not train_dir or not val_dir or not labels_csv:
    top_items = os.listdir(LOCAL_DATA)[:20]
    print(f"DEBUG — contents of {LOCAL_DATA}: {top_items}")

assert train_dir, f"Could not find train images in {LOCAL_DATA}"
assert val_dir, f"Could not find val images in {LOCAL_DATA}"
assert labels_csv, f"Could not find 4-class labels CSV in {LOCAL_DATA}"

def count_images(directory):
    total = 0
    for root, dirs, files in os.walk(directory):
        total += sum(1 for f in files if f.endswith(('.jpeg', '.jpg', '.png')))
    return total

print(f"\n✅ Data ready on local SSD:")
print(f"  Train: {train_dir} ({count_images(train_dir)} images)")
print(f"  Val:   {val_dir} ({count_images(val_dir)} images)")
print(f"  CSV:   {labels_csv}")

## 2. Model & Data Code (Self-Contained)

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from collections import Counter
import pandas as pd
from PIL import Image
import numpy as np
import random
import time
import json
from tqdm import tqdm
from sklearn.metrics import cohen_kappa_score, classification_report, confusion_matrix


# ==========================
# Seed
# ==========================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ==========================
# EfficientNet Backbone
# ==========================
class EfficientNetBackbone(nn.Module):
    VARIANTS = {
        "efficientnet_b0": (models.efficientnet_b0, models.EfficientNet_B0_Weights.IMAGENET1K_V1),
        "efficientnet_b3": (models.efficientnet_b3, models.EfficientNet_B3_Weights.IMAGENET1K_V1),
    }

    def __init__(self, variant="efficientnet_b3", pretrained=True):
        super().__init__()
        factory_fn, weights_enum = self.VARIANTS[variant]
        weights = weights_enum if pretrained else None
        model = factory_fn(weights=weights)
        self.feature_dim = model.classifier[1].in_features
        model.classifier = nn.Identity()
        self.model = model
        self.variant = variant

    def forward(self, x):
        return self.model(x)

    def get_finetune_layers(self):
        features = self.model.features
        return [features[-1], features[-2]]


# ==========================
# 4-Class Severity Head
# ==========================
class DRSeverityHead4Class(nn.Module):
    """4-class head: No Referable DR, Moderate, Severe, Proliferative"""
    def __init__(self, feature_dim, num_classes=4, dropout=0.4):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout * 0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, features):
        return self.classifier(features)


# ==========================
# E5 Model (backbone + 4-class head)
# ==========================
class E5Model(nn.Module):
    def __init__(self, backbone="efficientnet_b3", backbone_pretrained=True, num_classes=4):
        super().__init__()
        self.backbone = EfficientNetBackbone(variant=backbone, pretrained=backbone_pretrained)
        self.severity_head = DRSeverityHead4Class(
            feature_dim=self.backbone.feature_dim,
            num_classes=num_classes,
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.severity_head(features)


# ==========================
# Dataset (4-class)
# ==========================
class EyePACS4ClassDataset(Dataset):
    """EyePACS dataset with 4-class remapped labels."""

    # Remap: 0→0, 1→0, 2→1, 3→2, 4→3
    REMAP = {0: 0, 1: 0, 2: 1, 3: 2, 4: 3}
    CLASS_NAMES = ['No Referable DR', 'Moderate', 'Severe', 'Proliferative']

    def __init__(self, images_dir, labels_csv, transform=None, use_precomputed=True):
        """
        Args:
            images_dir: Path to split directory (contains DR/ and NORMAL/ subdirs)
            labels_csv: Path to labels CSV with 'image' and 'level' columns
            transform: torchvision transforms
            use_precomputed: If True, use 'level_4class' column if available
        """
        self.images_dir = Path(images_dir)
        self.transform = transform

        df = pd.read_csv(labels_csv)

        # Use precomputed 4-class labels if available, otherwise remap on-the-fly
        if use_precomputed and 'level_4class' in df.columns:
            self.label_map = {str(row['image']): int(row['level_4class']) for _, row in df.iterrows()}
            print("  Using precomputed level_4class column")
        else:
            self.label_map = {str(row['image']): self.REMAP[int(row['level'])] for _, row in df.iterrows()}
            print("  Remapping on-the-fly: 0+1→0, 2→1, 3→2, 4→3")

        valid_ext = {'.jpg', '.jpeg', '.png', '.ppm', '.bmp', '.tif', '.tiff'}
        self.samples = []
        for img_path in sorted(self.images_dir.rglob('*')):
            if img_path.suffix.lower() in valid_ext:
                img_id = img_path.stem
                if img_id in self.label_map:
                    self.samples.append((img_path, self.label_map[img_id]))

        # Distribution
        dist = Counter(s for _, s in self.samples)
        print(f"  Loaded {len(self.samples)} images from {images_dir}")
        for cls_id in sorted(dist.keys()):
            print(f"    Class {cls_id} ({self.CLASS_NAMES[cls_id]}): {dist[cls_id]}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, severity = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(severity, dtype=torch.long)

    def get_class_weights(self):
        """Compute inverse-frequency class weights for CrossEntropyLoss."""
        labels = [s for _, s in self.samples]
        dist = Counter(labels)
        total = len(labels)
        num_classes = len(self.CLASS_NAMES)
        weights = [total / (num_classes * dist.get(i, 1)) for i in range(num_classes)]
        return torch.tensor(weights, dtype=torch.float32)


print("✅ All model and data classes defined.")

## 3. Configuration

In [ ]:
# ==============================
# E5 HYPERPARAMETERS
# ==============================
BACKBONE = "efficientnet_b3"
IMAGE_SIZE = 384
BATCH_SIZE = 16           # T4 16GB can handle this at 384px
NUM_EPOCHS = 30           # More epochs since 4-class should converge better
UNFREEZE_EPOCH = 10       # Phase 2 starts here
LR_HEAD = 3e-4            # Slightly higher LR for head (fewer classes now)
LR_BACKBONE_FT = 1e-5    # 10x lower for backbone fine-tuning
WEIGHT_DECAY = 1e-4
SEED = 42
NUM_WORKERS = 2           # Colab has 2 CPU cores
NUM_CLASSES = 4

# Paths
TRAIN_DIR = train_dir
VAL_DIR = val_dir
LABELS_CSV = labels_csv
SAVE_DIR = os.path.join(DRIVE_PROJECT, "models/stage2_4class")
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Experiment:  E5 (4-class merged)")
print(f"Backbone:    {BACKBONE}")
print(f"Image size:  {IMAGE_SIZE}px")
print(f"Batch size:  {BATCH_SIZE}")
print(f"Epochs:      {NUM_EPOCHS} (unfreeze at {UNFREEZE_EPOCH})")
print(f"Classes:     {NUM_CLASSES} (No Referable DR, Moderate, Severe, Proliferative)")
print(f"Data:        local SSD ({LOCAL_DATA})")
print(f"Save dir:    {SAVE_DIR} (Google Drive)")

## 4. Data Loading

In [ ]:
set_seed(SEED)

normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225],
)

transform_train = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    normalize,
])

transform_eval = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    normalize,
])

print("Loading training data...")
train_dataset = EyePACS4ClassDataset(
    images_dir=TRAIN_DIR,
    labels_csv=LABELS_CSV,
    transform=transform_train,
)

print("\nLoading validation data...")
val_dataset = EyePACS4ClassDataset(
    images_dir=VAL_DIR,
    labels_csv=LABELS_CSV,
    transform=transform_eval,
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

# Compute class weights for imbalanced data
class_weights = train_dataset.get_class_weights()
print(f"\nClass weights: {class_weights.tolist()}")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

## 5. Model Initialization

In [ ]:
device = torch.device("cuda")

model = E5Model(backbone=BACKBONE, backbone_pretrained=True, num_classes=NUM_CLASSES)
model.to(device)

print(f"Backbone: {BACKBONE}")
print(f"Feature dim: {model.backbone.feature_dim}")
print(f"Output classes: {NUM_CLASSES}")

total_params = sum(p.numel() for p in model.parameters())
print(f"Total params: {total_params:,}")

# Freeze backbone for Phase 1
for param in model.backbone.parameters():
    param.requires_grad = False

model.backbone.eval()
model.severity_head.train()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 1 trainable: {trainable:,} / {total_params:,} ({100*trainable/total_params:.1f}%)")

## 6. Training Functions

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, epoch_num):
    model.severity_head.train()
    running_loss = 0.0
    num_batches = 0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc=f"Train E{epoch_num}", dynamic_ncols=True)
    for images, severity_labels in pbar:
        images = images.to(device)
        severity_labels = severity_labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, severity_labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        num_batches += 1

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(severity_labels.cpu().numpy())

        pbar.set_postfix({"loss": f"{running_loss/num_batches:.4f}"})

    train_qwk = cohen_kappa_score(all_labels, all_preds, weights="quadratic")
    return running_loss / num_batches, train_qwk


def validate(model, loader, criterion, device):
    model.eval()
    val_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, severity_labels in tqdm(loader, desc="Val", leave=False):
            images = images.to(device)
            severity_labels = severity_labels.to(device)

            logits = model(images)
            loss = criterion(logits, severity_labels)
            val_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(severity_labels.cpu().numpy())

    qwk = cohen_kappa_score(all_labels, all_preds, weights="quadratic")
    accuracy = np.mean(np.array(all_preds) == np.array(all_labels))

    # Per-class recall
    class_names = EyePACS4ClassDataset.CLASS_NAMES
    for cls_id, cls_name in enumerate(class_names):
        mask = np.array(all_labels) == cls_id
        if mask.sum() > 0:
            recall = (np.array(all_preds)[mask] == cls_id).mean()
        else:
            recall = 0.0
        # Store for printing outside

    return val_loss / len(loader), qwk, accuracy, all_preds, all_labels


def save_checkpoint(model, optimizer, epoch, path, **extra):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }
    checkpoint.update(extra)
    torch.save(checkpoint, path)


def unfreeze_top_blocks(model):
    """Unfreeze last 2 EfficientNet feature blocks."""
    blocks = model.backbone.get_finetune_layers()
    total_unfrozen = 0
    for block in blocks:
        for param in block.parameters():
            param.requires_grad = True
            total_unfrozen += param.numel()
        block.train()

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Unfrozen backbone params: {total_unfrozen:,}")
    print(f"Total trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
    return blocks


print("✅ Training functions defined.")

## 7. Training Loop

In [ ]:
# Class-weighted CrossEntropy for imbalanced classes
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

# Phase 1 optimizer: severity head only
optimizer = torch.optim.AdamW(
    model.severity_head.parameters(),
    lr=LR_HEAD,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
)

best_qwk = -1.0
history = []
start_time = time.time()
patience = 8  # Early stopping patience
no_improve = 0

CLASS_NAMES_SHORT = ['NoRef', 'Mod', 'Sev', 'Prolif']

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()

    # Phase transition
    if epoch == UNFREEZE_EPOCH:
        print(f"\n{'='*60}")
        print(f"Phase 2: Unfreezing top EfficientNet blocks (epoch {epoch+1})")
        print(f"{'='*60}")

        blocks = unfreeze_top_blocks(model)

        optimizer = torch.optim.AdamW([
            {"params": model.severity_head.parameters(), "lr": LR_HEAD * 0.5},
            *[{"params": block.parameters(), "lr": LR_BACKBONE_FT} for block in blocks],
        ], weight_decay=WEIGHT_DECAY)

        remaining = NUM_EPOCHS - epoch
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=remaining, eta_min=1e-6
        )
        no_improve = 0  # Reset patience on phase change

    phase_str = 'Phase 2' if epoch >= UNFREEZE_EPOCH else 'Phase 1'
    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}] ({phase_str})")

    train_loss, train_qwk = train_one_epoch(model, train_loader, optimizer, criterion, device, epoch+1)
    val_loss, val_qwk, val_acc, val_preds, val_labels = validate(model, val_loader, criterion, device)

    scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    epoch_time = time.time() - epoch_start

    # Per-class recall
    recalls = {}
    for cls_id, cls_name in enumerate(CLASS_NAMES_SHORT):
        mask = np.array(val_labels) == cls_id
        if mask.sum() > 0:
            recalls[cls_name] = (np.array(val_preds)[mask] == cls_id).mean()
        else:
            recalls[cls_name] = 0.0

    recall_str = " | ".join([f"{k}: {v:.1%}" for k, v in recalls.items()])

    print(f"  Train Loss: {train_loss:.4f} | Train QWK: {train_qwk:.4f}")
    print(f"  Val Loss:   {val_loss:.4f} | Val QWK: {val_qwk:.4f} | Val Acc: {val_acc:.4f}")
    print(f"  Recall → {recall_str}")
    print(f"  LR: {lr:.2e} | Time: {epoch_time:.0f}s")

    history.append({
        "epoch": epoch + 1,
        "train_loss": round(train_loss, 4),
        "train_qwk": round(train_qwk, 4),
        "val_loss": round(val_loss, 4),
        "val_qwk": round(val_qwk, 4),
        "val_acc": round(val_acc, 4),
        "recalls": {k: round(v, 4) for k, v in recalls.items()},
        "lr": lr,
        "time_sec": round(epoch_time, 1),
        "phase": 1 if epoch < UNFREEZE_EPOCH else 2,
    })

    if val_qwk > best_qwk:
        best_qwk = val_qwk
        no_improve = 0
        save_checkpoint(
            model, optimizer, epoch,
            path=os.path.join(SAVE_DIR, "best.pt"),
            backbone=BACKBONE, image_size=IMAGE_SIZE,
            num_classes=NUM_CLASSES,
            val_qwk=val_qwk, val_acc=val_acc,
            class_mapping="0+1→0, 2→1, 3→2, 4→3",
        )
        print(f"  ✅ New best model saved (QWK={val_qwk:.4f})")
    else:
        no_improve += 1
        if no_improve >= patience and epoch >= UNFREEZE_EPOCH:
            print(f"  ⚠️ Early stopping: no improvement for {patience} epochs")
            break

    # Save latest + history every epoch
    save_checkpoint(
        model, optimizer, epoch,
        path=os.path.join(SAVE_DIR, "latest.pt"),
        backbone=BACKBONE, image_size=IMAGE_SIZE,
        num_classes=NUM_CLASSES,
        val_qwk=val_qwk, val_acc=val_acc,
    )
    with open(os.path.join(SAVE_DIR, "training_history.json"), "w") as f:
        json.dump(history, f, indent=2)

total_time = time.time() - start_time
print(f"\n{'='*60}")
print(f"Training Complete! Best QWK: {best_qwk:.4f}")
print(f"Total time: {total_time/60:.1f} minutes")
print(f"Model saved to: {SAVE_DIR}")

## 8. Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs_list = [h['epoch'] for h in history]
train_losses = [h['train_loss'] for h in history]
val_losses = [h['val_loss'] for h in history]
qwks = [h['val_qwk'] for h in history]
accs = [h['val_acc'] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('E5: 4-Class DR Severity (Merged Classes 0+1)', fontsize=14, fontweight='bold')

# Loss curves
axes[0].plot(epochs_list, train_losses, 'b-o', label='Train Loss', markersize=4)
axes[0].plot(epochs_list, val_losses, 'r-o', label='Val Loss', markersize=4)
axes[0].axvline(x=UNFREEZE_EPOCH, color='green', linestyle='--', alpha=0.7, label='Phase 2 Start')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# QWK
axes[1].plot(epochs_list, qwks, 'g-o', markersize=4)
axes[1].axvline(x=UNFREEZE_EPOCH, color='green', linestyle='--', alpha=0.7, label='Phase 2 Start')
axes[1].axhline(y=0.6456, color='orange', linestyle='--', alpha=0.7, label='E3+ 5-class (0.6456)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('QWK')
axes[1].set_title('Quadratic Weighted Kappa'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Per-class recalls over time
for cls_name in CLASS_NAMES_SHORT:
    cls_recalls = [h['recalls'].get(cls_name, 0) for h in history]
    axes[2].plot(epochs_list, cls_recalls, '-o', markersize=3, label=cls_name)
axes[2].axvline(x=UNFREEZE_EPOCH, color='green', linestyle='--', alpha=0.7, label='Phase 2 Start')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Recall')
axes[2].set_title('Per-Class Recall'); axes[2].legend(); axes[2].grid(True, alpha=0.3)
axes[2].set_ylim([0, 1])

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_curves_e5.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Best QWK: {max(qwks):.4f} at epoch {epochs_list[np.argmax(qwks)]}")

## 9. Final Evaluation on Best Model

In [ ]:
import seaborn as sns

# Load best checkpoint
best_path = os.path.join(SAVE_DIR, "best.pt")
checkpoint = torch.load(best_path, map_location=device, weights_only=False)

model_eval = E5Model(backbone=BACKBONE, backbone_pretrained=False, num_classes=NUM_CLASSES)
model_eval.load_state_dict(checkpoint["model_state_dict"])
model_eval.to(device)
model_eval.eval()

print(f"Loaded best checkpoint from epoch {checkpoint['epoch'] + 1}")
print(f"Checkpoint QWK: {checkpoint.get('val_qwk', 'N/A')}")

# Run validation
_, final_qwk, final_acc, all_preds, all_labels = validate(
    model_eval, val_loader, criterion, device
)

print(f"\n{'='*60}")
print(f"E5 FINAL EVALUATION (4-Class Merged)")
print(f"{'='*60}")
print(f"QWK:      {final_qwk:.4f}")
print(f"Accuracy: {final_acc:.4f}")

# Classification report
class_names_full = ['No Referable DR (0)', 'Moderate (1)', 'Severe (2)', 'Proliferative (3)']
print(f"\n{classification_report(all_labels, all_preds, target_names=class_names_full, zero_division=0)}")

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names_full, yticklabels=class_names_full, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'E5 — 4-Class Confusion Matrix (QWK={final_qwk:.4f})')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix_e5.png'), dpi=150, bbox_inches='tight')
plt.show()

## 10. Comparison with Previous Experiments

| Experiment | Backbone | Classes | Loss | QWK | Mild Recall |
|---|---|---|---|---|---|
| E0 | ResNet18 | 5-class | CE | 0.6269 | 0% |
| E3+ | ResNet18 | 5-class | CE | 0.6456 | 0% |
| E4 | EffNet-B3 | 5-class | CE | 0.5919 | 0% |
| E4b | EffNet-B3 | 5-class | EMD | 0.6115 | 0% |
| **E5** | **EffNet-B3** | **4-class (merged)** | **Weighted CE** | **see above** | **N/A** |

In [ ]:
# Save final summary
summary = {
    "experiment": "E5",
    "description": "4-class DR severity with merged No DR + Mild",
    "backbone": BACKBONE,
    "image_size": IMAGE_SIZE,
    "num_classes": NUM_CLASSES,
    "class_mapping": {
        "0": "No Referable DR (old 0+1)",
        "1": "Moderate (old 2)",
        "2": "Severe (old 3)",
        "3": "Proliferative (old 4)",
    },
    "best_qwk": round(final_qwk, 4),
    "best_accuracy": round(final_acc, 4),
    "total_epochs_trained": len(history),
    "best_epoch": checkpoint['epoch'] + 1,
    "previous_best": {"experiment": "E3+", "qwk": 0.6456, "classes": 5},
}

with open(os.path.join(SAVE_DIR, "e5_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("Summary saved to:", os.path.join(SAVE_DIR, "e5_summary.json"))
print(f"\n✅ E5 complete! Download best.pt from: {SAVE_DIR}")